In [3]:
!pip install pandas numpy

   ---------------------------------------- 0.0/9.7 MB ? eta -:--:--
   - -------------------------------------- 0.3/9.7 MB ? eta -:--:--
   --- ------------------------------------ 0.8/9.7 MB 2.5 MB/s eta 0:00:04
   ---- ----------------------------------- 1.0/9.7 MB 2.1 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.7 MB 1.7 MB/s eta 0:00:05
   ----- ---------------------------------- 1.3/9.7 MB 1.7 MB/s eta 0:00:05
   ------ --------------------------------- 1.6/9.7 MB 1.4 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.7 MB 1.4 MB/s eta 0:00:06
   ------ --------------------------------- 1.6/9.7 MB 1.4 MB/s eta 0:00:06
   ------- -------------------------------- 1.8/9.7 MB 1.0 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.7 MB 1.0 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.7 MB 1.0 MB/s eta 0:00:08
   ------- -------------------------------- 1.8/9.7 MB 1.0 MB/s eta 0:00:08
   -------- --------------

In [6]:
!pip install scikit-learn==1.5.2

  Using cached scipy-1.17.1-cp313-cp313-win_amd64.whl.metadata (60 kB)
  Using cached joblib-1.5.3-py3-none-any.whl.metadata (5.5 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   ---------------------------------------- 0.0/11.0 MB ? eta -:--:--
   - -------------------------------------- 0.5/11.0 MB 1.6 MB/s eta 0:00:07
   --- ------------------------------------ 1.0/11.0 MB 2.0 MB/s eta 0:00:05
   ---- ----------------------------------- 1.3/11.0 MB 2.0 MB/s eta 0:00:05
   ------ --------------------------------- 1.8/11.0 MB 1.9 MB/s eta 0:00:05
   ------- -------------------------------- 2.1/11.0 MB 1.9 MB/s eta 0:00:05
   -------- ------------------------------- 2.4/11.0 MB 1.8 MB/s eta 0:00:05
   --------- ------------------------------ 2.6/11.0 MB 1.7 MB/s eta 0:00:05
   --------- ------------------------------ 2.6/11.0 MB 1.7 MB/s eta 0:00:05
   ---------- ---------------------------

In [13]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import pandas as pd
import numpy as np
import copy
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import average_precision_score, roc_auc_score, precision_recall_curve
from torch_geometric.nn import TransformerConv
from torch_geometric.nn.models.tgn import (
    TGNMemory, IdentityMessage, LastAggregator, LastNeighborLoader
)
from torch.nn import Linear, LayerNorm
from torch_geometric.loader import TemporalDataLoader
from torch_geometric.data import TemporalData

In [26]:

# ── 1. Загрузка и подготовка CSV ──────────────────────────────────────────────

def load_csv(path: str):
    df = pd.read_csv(path)

    # Парсим дату
    df['transferdt'] = pd.to_datetime(df['transferdt'])

    # Добавляем признак наличия даты до удаления строк
    df['has_date'] = df['transferdt'].notna().astype(int)

    # Удаляем строки с пропущенным временем
    before = len(df)
    df = df.dropna(subset=['transferdt'])
    print(f"Удалено строк с NaT: {before - len(df)}")

    # Сортировка по времени — критично для TGN
    df = df.sort_values('transferdt').reset_index(drop=True)

    # Энкодим все узлы вместе — src и dst в одном пространстве
    enc = LabelEncoder()
    all_nodes = pd.concat([df['sender_epk_id'], df['recipient_epk_id']])
    enc.fit(all_nodes)
    df['src'] = enc.transform(df['sender_epk_id'])
    df['dst'] = enc.transform(df['recipient_epk_id'])

    print(f"Уникальных узлов: {len(enc.classes_)}")
    print(f"src min/max: {df['src'].min()} / {df['src'].max()}")
    print(f"dst min/max: {df['dst'].min()} / {df['dst'].max()}")

    # Время в секунды от начала — float
    min_t = df['transferdt'].min()
    df['t'] = (df['transferdt'] - min_t).dt.total_seconds().astype(float)

    # Признаки ребра
    df['amount_log']          = np.log1p(df['amount'])
    df['hour']                = df['transferdt'].dt.hour / 23.0
    df['dayofweek']           = df['transferdt'].dt.dayofweek / 6.0
    df['sender_delta_t_log']  = np.log1p(
        df.groupby('src')['t'].diff().fillna(0)
    )
    df['sender_tx_count']     = df.groupby('src').cumcount()

    msg_cols = [
        'amount_log', 'hour', 'dayofweek',
        'sender_delta_t_log', 'sender_tx_count', 'has_date'
    ]

    return df, msg_cols, enc


def make_temporal_data(df, msg_cols):
    src = torch.tensor(df['src'].values, dtype=torch.long)
    dst = torch.tensor(df['dst'].values, dtype=torch.long)
    t   = torch.tensor(df['t'].values,   dtype=torch.float)
    msg = torch.tensor(df[msg_cols].values, dtype=torch.float)
    y   = torch.tensor(df['bad_flag'].values, dtype=torch.float)
    return TemporalData(src=src, dst=dst, t=t, msg=msg, y=y)



In [15]:

# ── 2. Хронологическое разбиение ──────────────────────────────────────────────

def split_data(df, val_ratio=0.15, test_ratio=0.15):
    n         = len(df)
    train_end = int(n * (1 - val_ratio - test_ratio))
    val_end   = int(n * (1 - test_ratio))

    df_train = df.iloc[:train_end].copy()
    df_val   = df.iloc[train_end:val_end].copy()
    df_test  = df.iloc[val_end:].copy()

    print(f"Train: {len(df_train):>8,} транзакций | фрод: {df_train['bad_flag'].mean():.2%}")
    print(f"Val:   {len(df_val):>8,} транзакций | фрод: {df_val['bad_flag'].mean():.2%}")
    print(f"Test:  {len(df_test):>8,} транзакций | фрод: {df_test['bad_flag'].mean():.2%}")

    return df_train, df_val, df_test


def normalize_features(df_train, df_val, df_test, msg_cols):
    mean = df_train[msg_cols].mean()
    std  = df_train[msg_cols].std().replace(0, 1)

    df_train[msg_cols] = (df_train[msg_cols] - mean) / std
    df_val[msg_cols]   = (df_val[msg_cols]   - mean) / std
    df_test[msg_cols]  = (df_test[msg_cols]  - mean) / std

    return df_train, df_val, df_test



In [ ]:

# ── 3. Архитектура ────────────────────────────────────────────────────────────

class GraphAttentionEmbedding(nn.Module):
    def __init__(self, in_channels, out_channels, msg_dim, heads=4, dropout=0.1):
        super().__init__()
        assert out_channels % heads == 0, "out_channels должен делиться на heads"
        self.conv = TransformerConv(
            in_channels,
            out_channels // heads,
            heads=heads,
            dropout=dropout,
            edge_dim=msg_dim,
            beta=True,
        )
        self.norm    = LayerNorm(out_channels)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, edge_index, msg):
        out = self.conv(x, edge_index, msg)
        return self.dropout(self.norm(out))


class EdgeClassifier(nn.Module):
    def __init__(self, node_dim, edge_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.norm = LayerNorm(node_dim * 2 + edge_dim)
        self.mlp  = nn.Sequential(
            Linear(node_dim * 2 + edge_dim, hidden_dim),
            LayerNorm(hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            Linear(hidden_dim, hidden_dim // 2),
            LayerNorm(hidden_dim // 2),
            nn.ReLU(),
            nn.Dropout(dropout),
            Linear(hidden_dim // 2, 1),
        )
        # sigmoid(-4.6) ≈ 0.01, соответствует ~1% доле фрода
        nn.init.constant_(self.mlp[-1].bias, -4.6)

    def forward(self, z_src, z_dst, edge_attr):
        x = torch.cat([z_src, z_dst, edge_attr], dim=-1)
        return self.mlp(self.norm(x))


In [ ]:
# ── 4. Сборка модели ──────────────────────────────────────────────────────────
import types

def build_model(num_nodes, msg_dim, device,
                memory_dim=128, embedding_dim=128, heads=4):

    memory = TGNMemory(
        num_nodes,
        msg_dim,
        memory_dim,
        time_dim=64,
        message_module=IdentityMessage(msg_dim, memory_dim, 64),
        aggregator_module=LastAggregator(),
    ).to(device)

    # Патч 1: меняем тип буфера last_update с long на float,
    # чтобы хранить float-метки времени из batch.t
    memory.register_buffer(
        'last_update',
        torch.zeros(num_nodes, dtype=torch.float32, device=device)
    )

    # Патч 2: в torch_geometric 2.7.0 _get_updated_memory при пустом
    # хранилище сообщений возвращает last_update как Long (баг библиотеки).
    # Перехватываем и явно приводим к float перед записью в буфер.
    def _patched_update_memory(self, n_id):
        memory_val, last_update = self._get_updated_memory(n_id)
        self.memory[n_id] = memory_val
        self.last_update[n_id] = last_update.float()

    memory._update_memory = types.MethodType(_patched_update_memory, memory)

    gnn = GraphAttentionEmbedding(
        in_channels  = memory_dim,
        out_channels = embedding_dim,
        msg_dim      = msg_dim,
        heads        = heads,
    ).to(device)

    classifier = EdgeClassifier(
        node_dim = embedding_dim,
        edge_dim = msg_dim,
    ).to(device)

    neighbor_loader = LastNeighborLoader(num_nodes, size=15, device=device)

    return memory, gnn, classifier, neighbor_loader


In [ ]:

# ── 5. Focal Loss ─────────────────────────────────────────────────────────────

class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        bce     = F.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        pt      = torch.exp(-bce)
        alpha_t = targets * self.alpha + (1 - targets) * (1 - self.alpha)
        focal   = alpha_t * (1 - pt) ** self.gamma * bce
        return focal.mean()


criterion = FocalLoss(alpha=0.75, gamma=2.0)


In [19]:
# ── 6. Train ──────────────────────────────────────────────────────────────────

def train(memory, gnn, classifier, neighbor_loader,
          train_loader, optimizer, device, all_msg):
    memory.train()
    gnn.train()
    classifier.train()

    memory.reset_state()
    neighbor_loader.reset_state()

    total_loss, total_examples = 0, 0

    for batch in train_loader:
        batch = batch.to(device)
        optimizer.zero_grad()

        # Все уникальные узлы батча
        n_id = torch.cat([batch.src, batch.dst]).unique()
        n_id, edge_index, e_id = neighbor_loader(n_id)

        z, last_update = memory(n_id)

        # e_id — глобальные индексы из neighbor_loader, индексируем all_msg
        z = gnn(z, edge_index, all_msg[e_id])

        assoc = torch.empty(
            n_id.max() + 1, dtype=torch.long, device=device
        )
        assoc[n_id] = torch.arange(n_id.size(0), device=device)

        logits = classifier(
            z[assoc[batch.src]],
            z[assoc[batch.dst]],
            batch.msg,
        ).squeeze(-1)

        loss = criterion(logits, batch.y.float())
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            list(memory.parameters()) +
            list(gnn.parameters()) +
            list(classifier.parameters()),
            max_norm=1.0,
        )
        optimizer.step()

        memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        neighbor_loader.insert(batch.src, batch.dst)

        # Обрываем вычислительный граф после каждого батча
        memory.memory = memory.memory.detach()
        memory.last_update = memory.last_update.detach()

        total_loss     += float(loss) * batch.num_events
        total_examples += batch.num_events

    return total_loss / total_examples


In [20]:
# ── 7. Test ───────────────────────────────────────────────────────────────────

@torch.no_grad()
def test(memory, gnn, classifier, neighbor_loader, loader, device, all_msg):
    memory.eval()
    gnn.eval()
    classifier.eval()

    # Backup — чтобы test() был идемпотентен
    mem_backup = copy.deepcopy(memory.state_dict())
    nl_backup  = copy.deepcopy(neighbor_loader)

    all_logits, all_labels = [], []

    for batch in loader:
        batch = batch.to(device)

        n_id = torch.cat([batch.src, batch.dst]).unique()
        n_id, edge_index, e_id = neighbor_loader(n_id)

        z, last_update = memory(n_id)
        z = gnn(z, edge_index, all_msg[e_id])

        assoc = torch.empty(
            n_id.max() + 1, dtype=torch.long, device=device
        )
        assoc[n_id] = torch.arange(n_id.size(0), device=device)

        logits = classifier(
            z[assoc[batch.src]],
            z[assoc[batch.dst]],
            batch.msg,
        ).squeeze(-1)

        all_logits.append(logits.cpu())
        all_labels.append(batch.y.cpu())

        memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        neighbor_loader.insert(batch.src, batch.dst)

    # Восстанавливаем состояние памяти
    memory.load_state_dict(mem_backup)
    neighbor_loader.__dict__.update(nl_backup.__dict__)

    probs  = torch.sigmoid(torch.cat(all_logits)).numpy()
    labels = torch.cat(all_labels).numpy()

    ap  = average_precision_score(labels, probs)
    auc = roc_auc_score(labels, probs)

    return ap, auc, probs, labels


In [21]:
# ── 8. Inference ──────────────────────────────────────────────────────────────

@torch.no_grad()
def inference(memory, gnn, classifier, neighbor_loader, loader, device, all_msg):
    memory.eval()
    gnn.eval()
    classifier.eval()

    all_probs = []

    for batch in loader:
        batch = batch.to(device)

        n_id = torch.cat([batch.src, batch.dst]).unique()
        n_id, edge_index, e_id = neighbor_loader(n_id)

        z, last_update = memory(n_id)
        z = gnn(z, edge_index, all_msg[e_id])

        assoc = torch.empty(
            n_id.max() + 1, dtype=torch.long, device=device
        )
        assoc[n_id] = torch.arange(n_id.size(0), device=device)

        logits = classifier(
            z[assoc[batch.src]],
            z[assoc[batch.dst]],
            batch.msg,
        ).squeeze(-1)

        all_probs.append(torch.sigmoid(logits).cpu())

        memory.update_state(batch.src, batch.dst, batch.t, batch.msg)
        neighbor_loader.insert(batch.src, batch.dst)

        memory.memory      = memory.memory.detach()
        memory.last_update = memory.last_update.detach()

    return torch.cat(all_probs).numpy()


def find_threshold(labels, probs, target_recall=0.90):
    precision, recall, thresholds = precision_recall_curve(labels, probs)
    mask = recall[:-1] >= target_recall
    if not mask.any():
        return 0.5
    return float(thresholds[mask][-1])


In [24]:
# ── 9. Главный цикл ───────────────────────────────────────────────────────────

def main(csv_path: str):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print(f"Device: {device}")

    # Данные
    df, msg_cols, enc = load_csv(csv_path)
    print("Load CSV")
    df_train, df_val, df_test = split_data(df)
    print("Split data")
    df_train, df_val, df_test = normalize_features(
        df_train, df_val, df_test, msg_cols
    )
    print("Normalize features")

    data_train = make_temporal_data(df_train, msg_cols)
    data_val   = make_temporal_data(df_val,   msg_cols)
    data_test  = make_temporal_data(df_test,  msg_cols)

    train_loader = TemporalDataLoader(data_train, batch_size=200)
    val_loader   = TemporalDataLoader(data_val,   batch_size=200)
    test_loader  = TemporalDataLoader(data_test,  batch_size=200)

    # Единый тензор сообщений в порядке train → val → test.
    # LastNeighborLoader хранит глобальные индексы рёбер, накапливая их
    # через все батчи и фазы, поэтому e_id может ссылаться на любое
    # ранее виденное ребро. all_msg[e_id] всегда даёт правильное сообщение.
    all_msg = torch.cat([data_train.msg, data_val.msg, data_test.msg], dim=0).to(device)

    # Модель
    # num_nodes по всему датасету до разбиения
    num_nodes = len(enc.classes_)
    msg_dim   = len(msg_cols)

    memory, gnn, classifier, neighbor_loader = build_model(
        num_nodes = num_nodes,
        msg_dim   = msg_dim,
        device    = device,
    )

    optimizer = torch.optim.Adam(
        list(memory.parameters()) +
        list(gnn.parameters()) +
        list(classifier.parameters()),
        lr=1e-4,
    )
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=5
    )

    best_val_ap = 0
    best_state  = None
    no_improve  = 0
    patience    = 10

    for epoch in range(1, 101):
        loss = train(
            memory, gnn, classifier, neighbor_loader,
            train_loader, optimizer, device, all_msg,
        )

        val_ap, val_auc, val_probs, val_labels = test(
            memory, gnn, classifier, neighbor_loader,
            val_loader, device, all_msg,
        )

        scheduler.step(val_ap)

        print(f"Epoch {epoch:03d} | loss {loss:.4f} | "
              f"val AP {val_ap:.4f} | val AUC {val_auc:.4f}")

        if val_ap > best_val_ap:
            best_val_ap = val_ap
            best_state  = {
                'memory':     copy.deepcopy(memory.state_dict()),
                'gnn':        copy.deepcopy(gnn.state_dict()),
                'classifier': copy.deepcopy(classifier.state_dict()),
            }
            no_improve = 0
        else:
            no_improve += 1
            if no_improve >= patience:
                print(f"Early stopping на эпохе {epoch}")
                break

    # Загружаем лучшие веса
    memory.load_state_dict(best_state['memory'])
    gnn.load_state_dict(best_state['gnn'])
    classifier.load_state_dict(best_state['classifier'])

    # Прогоняем val и получаем предсказания за один проход.
    # Второй вызов был лишним: он дублировал вставку val-рёбер в neighbor_loader,
    # сдвигая счётчик e_id за границы all_msg при последующем тесте.
    probs_val = inference(
        memory, gnn, classifier, neighbor_loader, val_loader, device, all_msg
    )
    threshold = find_threshold(
        df_val['bad_flag'].values, probs_val, target_recall=0.90
    )
    print(f"\nПодобранный порог: {threshold:.3f}")

    # Финальный тест
    probs_test = inference(
        memory, gnn, classifier, neighbor_loader, test_loader, device, all_msg
    )

    test_ap  = average_precision_score(df_test['bad_flag'].values, probs_test)
    test_auc = roc_auc_score(df_test['bad_flag'].values, probs_test)
    preds    = (probs_test >= threshold).astype(int)

    print(f"Test AP:  {test_ap:.4f}")
    print(f"Test AUC: {test_auc:.4f}")
    print(f"Подозрительных: {preds.sum()} из {len(preds)} "
          f"({preds.mean():.2%})")

    # Сохраняем результаты
    df_result = df_test.copy()
    df_result['fraud_prob']      = probs_test
    df_result['predicted_fraud'] = preds
    df_result.to_csv('predictions.csv', index=False)
    print("Результаты сохранены в predictions.csv")

    torch.save(best_state, 'tgn_best.pt')
    print("Модель сохранена в tgn_best.pt")


In [27]:

if __name__ == '__main__':
    main('preprocessing_data.csv')

Device: cpu
Удалено строк с NaT: 2398
Уникальных узлов: 500
src min/max: 0 / 499
dst min/max: 0 / 499
Load CSV
Train:    5,321 транзакций | фрод: 1.15%
Val:      1,140 транзакций | фрод: 1.14%
Test:     1,141 транзакций | фрод: 0.70%
Split data
Normalize features


RuntimeError: Index put requires the source and destination dtypes match, got Float for the destination and Long for the source.

In [1]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

np.random.seed(42)

n_transactions = 10000
n_accounts     = 500

# Генерируем id счетов в формате EPK
account_ids = [f"EPK_{i:06d}" for i in range(n_accounts)]

# Временной диапазон — 6 месяцев
start_date = datetime(2024, 1, 1)
end_date   = datetime(2024, 6, 30)
date_range = (end_date - start_date).total_seconds()

# Генерируем транзакции
senders    = np.random.choice(account_ids, size=n_transactions)
recipients = np.random.choice(account_ids, size=n_transactions)

# Отправитель != получатель
mask = senders == recipients
while mask.any():
    recipients[mask] = np.random.choice(account_ids, size=mask.sum())
    mask = senders == recipients

# Суммы — логнормальное распределение (реалистично для транзакций)
amounts = np.random.lognormal(mean=7, sigma=1.5, size=n_transactions)
amounts = np.round(amounts, 2)

# Временные метки — равномерно + ~24% NaT как в ваших данных
timestamps = [
    start_date + timedelta(seconds=np.random.uniform(0, date_range))
    for _ in range(n_transactions)
]
timestamps = sorted(timestamps)  # сортируем по времени

# Добавляем ~24% пропусков в даты
nat_mask = np.random.random(n_transactions) < 0.24
timestamps_with_nat = [
    None if nat_mask[i] else timestamps[i]
    for i in range(n_transactions)
]

# Генерируем bad_flag (~1.5% фрода)
# Фрод коррелирует с большими суммами и частыми транзакциями
bad_flag = np.zeros(n_transactions, dtype=bool)

# Случайный фрод
random_fraud = np.random.random(n_transactions) < 0.008
bad_flag |= random_fraud

# Фрод на больших суммах
high_amount_fraud = (amounts > np.percentile(amounts, 95)) & \
                    (np.random.random(n_transactions) < 0.05)
bad_flag |= high_amount_fraud

# Несколько мошеннических аккаунтов — много транзакций подряд
fraud_accounts = np.random.choice(account_ids, size=5)
fraud_mask = np.isin(senders, fraud_accounts) & \
             (np.random.random(n_transactions) < 0.15)
bad_flag |= fraud_mask

df = pd.DataFrame({
    'sender_epk_id':    senders,
    'recipient_epk_id': recipients,
    'amount':           amounts,
    'transferdt':       timestamps_with_nat,
    'bad_flag':         bad_flag,
})

print(f"Всего транзакций: {len(df):,}")
print(f"Фрод: {df['bad_flag'].sum():,} ({df['bad_flag'].mean():.2%})")
print(f"NaT в transferdt: {df['transferdt'].isna().sum():,} "
      f"({df['transferdt'].isna().mean():.2%})")
print(f"\nПример данных:")
print(df.head(10).to_string())

df.to_csv('preprocessing_data.csv', index=False)
print("\nФайл сохранён: preprocessing_data.csv")

Всего транзакций: 10,000
Фрод: 118 (1.18%)
NaT в transferdt: 2,398 (23.98%)

Пример данных:
  sender_epk_id recipient_epk_id    amount                 transferdt  bad_flag
0    EPK_000102       EPK_000181   1542.90 2024-01-01 00:05:20.868080     False
1    EPK_000435       EPK_000012   2610.84 2024-01-01 00:26:40.942508     False
2    EPK_000348       EPK_000338   1930.56                        NaT     False
3    EPK_000270       EPK_000284    750.72 2024-01-01 00:52:29.544584     False
4    EPK_000106       EPK_000258    490.93 2024-01-01 01:04:38.693963     False
5    EPK_000071       EPK_000089    452.89 2024-01-01 01:57:02.081537     False
6    EPK_000188       EPK_000103    273.72                        NaT     False
7    EPK_000020       EPK_000378    216.33                        NaT     False
8    EPK_000102       EPK_000185  17286.76                        NaT      True
9    EPK_000121       EPK_000249   1322.22 2024-01-01 03:41:18.715908     False

Файл сохранён: preprocessin